# Reaction-time models with permutation test

The reaction-time predictor is a deterministic function of 15 letter-level values that are the
same for every participant. A mixed model fitted to ~2600 trials treats those trials as
independent evidence about that predictor. This notebook
replaces the model's p-value with a permutation p-value that respects the 15-letter structure.

Two models are compared. They differ only in what goes in the predictor column:

| model | predictor for a trial |
|---|---|
| **continuous** | mean decoding accuracy of the letters in the string |
| **binary** | fraction of the string's letters that are *significantly* decoded (8 of 15, FDR) |

The 15 letter values are reassigned across letters, every trial's predictor is
recomputed, and the model is refitted. Because each null replicate is produced by the same model,
whatever inflation the model has appears in the null too and cancels. 


In [ ]:
import itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.rcParams.update({'font.size': 12})
LETTERS = ['B','C','D','F','G','H','K','L','N','P','R','S','T','V','Z']
L_IDX   = {l: i for i, l in enumerate(LETTERS)}

# ---- configuration -----------------------------------------------------------
N_PERM_CONT  = 5000     # Monte Carlo draws for the continuous model
EXACT_BINARY = True     # enumerate all 6435 labellings (False -> N_PERM_CONT draws)
SEED         = 0
# Full run is 6435 + 5000 model fits, roughly 20 min on one core.
# ------------------------------------------------------------------------------

## 1. Load data

`letters_sep` / `probes_sep` / `subjs_sep` contain **only correct trials**; `trial_corrInfo` contains all trials and is filtered to `col1 == 1` to align.

In [ ]:
info    = np.load('../Data/trial_corrInfo.npy', allow_pickle=True)
letters = np.load('../Data/letters_sep.npy',    allow_pickle=True)
probes  = np.load('../Data/probes_sep.npy',     allow_pickle=True)
subjs   = np.load('../Data/subjs_sep.npy',      allow_pickle=True)

subj    = np.concatenate(subjs, axis=0)
rawL    = [list(s) for s in np.concatenate(letters, axis=0)]
rawP    = [list(s) for s in np.concatenate(probes,  axis=0)]

infoALL = np.concatenate(info)
n_all, n_ok = len(infoALL), int((infoALL[:, 1] == 1).sum())
infoALL = infoALL[infoALL[:, 1] == 1]

assert len(rawL) == n_ok == len(subj)
print(f'trials: {n_all} total -> {n_ok} correct;  participants: {sorted(np.unique(subj))}')

## 2. Decoding accuracy and significance per letter

In [ ]:
T1, T2 = 2, 16

def load_decoding(n_splits=50):
    acc = [np.array([np.diag(np.load(f'../Results/single_letter_50_{k}.npy')[i])
                     for i in range(15)]) for k in range(n_splits)]
    acc = np.array(acc)
    return np.median(np.array([np.cumsum(acc[:, i, T1:T2], axis=-1)[:, -1] / (T2 - T1)
                               for i in range(15)]), axis=1)

DEC = load_decoding(); print('decoding accuracies loaded from ../Results/')

SIG_LETTERS = ['F','B','G','C','V','D','N','R']     # FDR p<.05, see 2c_letter_decoding
sig = np.array([l in SIG_LETTERS for l in LETTERS])
print(pd.DataFrame({'decoding_acc': DEC.round(4), 'significant': sig},
                   index=LETTERS).T.to_string())

## 3. Phonological covariates

In [ ]:
FEAT = np.array([
 [1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0],
 [1,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0],
 [1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0],
 [0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0],[1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,1,0],
 [1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0],[0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],
 [1,0,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0],[0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0],
 [0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1],
 [0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0]], dtype=float)
_N  = FEAT / np.linalg.norm(FEAT, axis=1, keepdims=True)
COS = _N @ _N.T

def mean_pairwise_cosine(ls):
    idx = [L_IDX[l] for l in ls]
    if len(idx) < 2: return np.nan
    return np.mean([COS[idx[a], idx[b]]
                    for a in range(len(idx)) for b in range(a+1, len(idx))])

def probe_string_cosine(p, ls):
    o = [l for l in ls if l != p]
    return np.mean([COS[L_IDX[p], L_IDX[l]] for l in o]) if o else np.nan

## 4. Trial dataframe

In [ ]:
rows = []
for i in range(len(infoALL)):
    raw_s, raw_p = rawL[i], rawP[i]
    ls = [c for c in raw_s if c in L_IDX]
    ps = [c for c in raw_p if c in L_IDX]
    p  = ps[0] if len(ps) else None
    inf = infoALL[i]
    r = {'rt': inf[4], 'trial_type': 'IN' if int(inf[3]) == 51 else 'OUT',
         'size': int(inf[0]), 'subject': subj[i], 'probe': p,
         'phon_sim': mean_pairwise_cosine(ls),
         'probe_phon_sim': probe_string_cosine(p, ls) if p else np.nan}
    for l in LETTERS:
        r['has_' + l] = float(l in ls)
    rows.append(r)

df = (pd.DataFrame(rows)
      .dropna(subset=['rt','probe','phon_sim','probe_phon_sim'])
      .reset_index(drop=True))
print(f'n = {len(df)} trials, {df.subject.nunique()} participants')

## 5. The two models

Identical apart from the predictor column, and identical to the published specification otherwise:

```
rt ~ predicted_c + phon_sim_c + probe_phon_sim_c
     + C(trial_type) + C(size) + C(probe) + (1 | subject)
```

In [ ]:
BASE = ('rt ~ predicted_c + phon_sim_c + probe_phon_sim_c '
        '+ C(trial_type) + C(size) + C(probe)')

def prep(data):
    d  = data.copy().reset_index(drop=True)
    H  = d[['has_' + l for l in LETTERS]].to_numpy()
    Hn = H / H.sum(1, keepdims=True)                 # row-normalised -> mean over the string
    for c in ['phon_sim', 'probe_phon_sim']:
        d[c + '_c'] = d[c] - d[c].mean()
    d['trial_type'] = pd.Categorical(d.trial_type, ['OUT', 'IN'])
    d['size']       = pd.Categorical(d['size'], sorted(d['size'].unique()))
    d['probe']      = pd.Categorical(d.probe, ['G'] + [l for l in LETTERS if l != 'G'])
    return d, Hn

def fit_z(d, Hn, values):
    """values: length-15, one number per letter. Returns (beta, z, nominal p)."""
    pv = Hn @ np.asarray(values, dtype=float)
    dd = d.copy(); dd['predicted_c'] = pv - pv.mean()
    r  = smf.mixedlm(BASE, dd, groups=dd['subject'], re_formula='1').fit(reml=False)
    b  = r.fe_params['predicted_c']
    return b, b / r.bse_fe['predicted_c'], r.pvalues['predicted_c']

# quick check that both models fit
for nm, v in [('continuous', DEC), ('binary', sig.astype(float))]:
    d, Hn = prep(df)
    b, z, p = fit_z(d, Hn, v)
    print(f'{nm:11s} beta = {b:+.4f} s   z = {z:+.3f}   nominal p = {p:.5f}')

## 6. Permutation

For the **continuous** model the 15 accuracies are randomly reassigned across letters
(`N_PERM_CONT` draws from the 15! possibilities). For the **binary** model every one of the 6435
ways to label 8 of 15 letters as decoded is enumerated, so the p-value is exact.

The p-value counts how many null replicates reach `|z|` at least as large as observed.

In [ ]:
ALL_SPLITS = [np.isin(np.arange(15), s)
              for s in itertools.combinations(range(15), 8)]

def permute(data, values, kind, label):
    d, Hn = prep(data)
    _, z_obs, p_nom = fit_z(d, Hn, values)

    rng = np.random.default_rng(SEED)
    if kind == 'binary' and EXACT_BINARY:
        draws, exact = [m.astype(float) for m in ALL_SPLITS], True
    elif kind == 'binary':
        draws = [rng.permutation(sig).astype(float) for _ in range(N_PERM_CONT)]; exact = False
    else:
        draws = [rng.permutation(values) for _ in range(N_PERM_CONT)]; exact = False

    t0, zn = time.time(), np.empty(len(draws))
    for k, v in enumerate(draws):
        zn[k] = fit_z(d, Hn, v)[1]
        if k == 24:
            print(f'  [{label}] {(time.time()-t0)/25:.3f}s/fit, '
                  f'ETA {len(draws)*(time.time()-t0)/25/60:.1f} min', flush=True)

    p2 = np.mean(np.abs(zn) >= abs(z_obs) - 1e-12)
    p1 = np.mean(zn <= z_obs + 1e-12)
    return {'label': label, 'n': len(d), 'z_obs': z_obs, 'p_nominal': p_nom,
            'n_perm': len(zn), 'exact': exact, 'null': zn,
            'null_sd': zn.std(), 'p_perm_2s': p2, 'p_perm_1s': p1,
            'n_extreme': int(np.sum(np.abs(zn) >= abs(z_obs) - 1e-12))}

In [ ]:
runs = []
for kind, values in [('continuous (as published)', DEC), ('binary (decoded / not)', sig.astype(float))]:
    print(f'running {kind} ...', flush=True)
    runs.append(permute(df, values, 'binary' if kind.startswith('binary') else 'continuous',
                        kind))
print('done')

## 7. Results

In [ ]:
summary = pd.DataFrame([{
    'model':          r['label'],
    'n trials':       r['n'],
    'z observed':     round(r['z_obs'], 3),
    'p nominal':      f"{r['p_nominal']:.5f}",
    'null SD':        round(r['null_sd'], 3),
    'n perm':         r['n_perm'],
    'exact':          r['exact'],
    'extreme':        r['n_extreme'],
    'p permutation':  round(r['p_perm_2s'], 4),
    'p perm 1-sided': round(r['p_perm_1s'], 4)} for r in runs])
print(summary.to_string(index=False))

import os
os.makedirs('../Results', exist_ok=True)
summary.to_csv('../Results/permutation_summary.csv', index=False)
np.save('../Results/permutation_nulls.npy',
        np.array([r['null'] for r in runs], dtype=object), allow_pickle=True)

## 8. Full fixed-effects tables

In [17]:
def fit_full(values):
    d, Hn = prep(df)
    pv = Hn @ np.asarray(values, dtype=float)
    d['predicted_c'] = pv - pv.mean()
    return smf.mixedlm(BASE, d, groups=d['subject'], re_formula='1').fit(reml=False)

RENAME = {'Intercept':        'Intercept',
          'phon_sim_c':       'Phonological similarity (string)',
          'probe_phon_sim_c': 'Phonological similarity (probe-string)',
          'C(trial_type)[T.IN]': 'IN vs. OUT',
          'C(size)[T.6]':     'Set size 6 vs. 4',
          'C(size)[T.8]':     'Set size 8 vs. 4'}

def clean(term, key_name):
    if term == 'predicted_c':
        return key_name
    if term in RENAME:
        return RENAME[term]
    if term.startswith('C(probe)[T.'):
        return term[len('C(probe)[T.'):-1] + ' vs. G'
    return term

def coef_table(r, key_name, perm_p):
    idx = r.fe_params.index
    ci  = r.conf_int().loc[idx]
    t = pd.DataFrame({
        'Coefficient': r.fe_params.round(3).values,
        '95% CI': [f'[{lo:.3f}, {hi:.3f}]' for lo, hi in zip(ci.iloc[:, 0], ci.iloc[:, 1])],
        'z': (r.fe_params / r.bse_fe).round(3).values,
        'p (model)': [f'{p:.2e}' for p in r.pvalues.loc[idx]],
        'p (permutation)': ['' for _ in idx]},
        index=[clean(t_, key_name) for t_ in idx])
    t.loc[key_name, 'p (permutation)'] = f'{perm_p:.4f}'

    priority = ['Intercept', key_name,
                'Phonological similarity (string)',
                'Phonological similarity (probe-string)',
                'IN vs. OUT', 'Set size 6 vs. 4', 'Set size 8 vs. 4']
    order = [x for x in priority if x in t.index] +             [x for x in t.index if x not in priority]
    return t.loc[order]

tables = {}
for r, values, key in [(runs[0], DEC, 'Decoding accuracy (continuous)'),
                       (runs[1], sig.astype(float), 'Decoding accuracy (binary)')]:
    m = fit_full(values)
    tab = coef_table(m, key, r['p_perm_2s'])
    tables[r['label']] = tab
    print(f"===== {r['label']}  (n = {r['n']}, {df.subject.nunique()} participants) =====")
    print(tab.to_string())
    print()
    tab.to_csv(f"../Results/fixed_effects_{'binary' if 'binary' in r['label'] else 'continuous'}.csv")

===== continuous (as published)  (n = 2585, 11 participants) =====
                                        Coefficient            95% CI       z p (model) p (permutation)
Intercept                                     1.448    [1.213, 1.683]  12.077  1.39e-33                
Decoding accuracy (continuous)               -2.646  [-4.875, -0.417]  -2.327  2.00e-02          0.1254
Phonological similarity (string)             -0.453  [-0.774, -0.132]  -2.764  5.72e-03                
Phonological similarity (probe-string)        0.533    [0.242, 0.824]   3.591  3.30e-04                
IN vs. OUT                                   -0.146  [-0.201, -0.090]  -5.145  2.67e-07                
Set size 6 vs. 4                              0.218    [0.153, 0.283]   6.543  6.03e-11                
Set size 8 vs. 4                              0.341    [0.272, 0.411]   9.612  7.11e-22                
B vs. G                                       0.100   [-0.055, 0.255]   1.264  2.06e-01              